## Kube-vip as Loadbalancer (Daemonset)

> Note :  You should run service first as loadbalancer to make one worker take ip

In [ ]:
kubectl apply -f https://kube-vip.io/manifests/rbac.yaml

export VIP=172.16.6.90
export INTERFACE=ens192
export KVVERSION=$(curl -sL https://api.github.com/repos/kube-vip/kube-vip/releases | jq -r ".[0].name")

alias kube-vip="ctr image pull ghcr.io/kube-vip/kube-vip:$KVVERSION; \
  ctr run --rm --net-host ghcr.io/kube-vip/kube-vip:$KVVERSION vip /kube-vip"

# kube-vip manifest daemonset \
#     --interface $INTERFACE \
#     --address $VIP \
#     --inCluster \
#     --services \
#     --arp \
#     --leaderElection > kube-vip-worker-lb-ds.yaml

kube-vip manifest daemonset \
    --interface $INTERFACE \
    --inCluster \
    --services \
    --arp \
    --leaderElection > kube-vip-worker-lb-ds.yaml

In [ ]:
vim kube-vip-worker-lb-ds.yaml

In [ ]:
metadata:
  name: kube-vip-worker-ds        # ← Rename to avoid conflict with existing DS
	
# Inside the kube-vip-lb-ds.yaml file, under spec.template.spec.
    spec:
      # ── ADD THIS: Run only on workers ──────────────────────────────
      affinity:
        nodeAffinity:
          requiredDuringSchedulingIgnoredDuringExecution:
            nodeSelectorTerms:
            - matchExpressions:
              - key: node-role.kubernetes.io/control-plane
                operator: DoesNotExist

# Inside the kube-vip-lb-ds.yaml file, under spec.template.spec.containers[0].env
        - name: svc_election
          value: "true"
        - name: cp_enable
          value: "false"
        - name: svc_leasename
  				value: plndr-svcs-worker-lock   # ← change this
				- name: vip_leaderelection
					value: "false" 

In [ ]:
kubectl apply -f kube-vip-worker-lb-ds.yaml

Show data

In [ ]:
kubectl get ds -n kube-system -o wide
kubectl get pods -n kube-system -l app.kubernetes.io/name=kube-vip-ds -o wide

Edit

In [ ]:
kubectl rollout restart ds kube-vip-ds -n kube-system

# Verify the new lease appears (held by a worker this time)
kubectl get lease -n kube-system | grep worker

# Watch for the IP
kubectl get svc test-nginx -w

In [ ]:
kubectl delete ds kube-vip-ds -n kube-system --force --grace-period=0

## Ingress

In [ ]:
helm repo add ingress-nginx https://kubernetes.github.io/ingress-nginx
helm repo update

In [ ]:
vim nginx-ingress-values.yaml

In [ ]:
controller:
  replicaCount: 1

  hostNetwork: true
  dnsPolicy: ClusterFirstWithHostNet

  hostPort:
    enabled: true
    ports:
      http: 80
      https: 443

  service:
    type: LoadBalancer
    loadBalancerIP: 172.16.6.90

  admissionWebhooks:
    enabled: false

  metrics:
    enabled: true

  podDisruptionBudget:
    enabled: true

In [ ]:
helm upgrade --install ingress-nginx ingress-nginx/ingress-nginx \
  --namespace ingress-nginx \
  --create-namespace \
  -f nginx-ingress-values.yaml

If change the value file

In [ ]:
helm upgrade ingress-nginx ingress-nginx/ingress-nginx \
  --namespace ingress-nginx \
  -f nginx-ingress-values.yaml

show if

In [ ]:
kubectl get svc -n ingress-nginx -w

In [ ]:
kubectl rollout status deployment ingress-nginx-controller -n ingress-nginx

---

`Old data`

In [ ]:
kubectl apply -f https://raw.githubusercontent.com/kubernetes/ingress-nginx/main/deploy/static/provider/baremetal/deploy.yaml

kubectl delete validatingwebhookconfiguration ingress-nginx-admission

In [ ]:
cat <<EOF | kubectl apply -f -
apiVersion: v1
kind: Service
metadata:
  name: ingress-nginx-controller
  namespace: ingress-nginx
  annotations:
    kube-vip.io/loadbalancerIPs: "172.16.6.90"
spec:
  type: LoadBalancer
  externalTrafficPolicy: Local
  ports:
    - name: http
      port: 80
      targetPort: 80
      protocol: TCP
    - name: https
      port: 443
      targetPort: 443
      protocol: TCP
  selector:
    app.kubernetes.io/name: ingress-nginx
    app.kubernetes.io/component: controller
EOF

In [ ]:
kubectl annotate svc ingress-nginx-controller \
  -n ingress-nginx \
  kube-vip.io/loadbalancerIPs=172.16.6.90

In [ ]:
kubectl patch deployment ingress-nginx-controller -n ingress-nginx --patch '
spec:
  template:
    spec:
      containers:
      - name: controller
        ports:
        - containerPort: 80
          hostPort: 80
          protocol: TCP
        - containerPort: 443
          hostPort: 443
          protocol: TCP'

For test

In [ ]:
# kubectl rollout status deployment ingress-nginx-controller -n ingress-nginx
# kubectl edit svc -n ingress-nginx ingress-nginx-controller

kubectl get svc -n ingress-nginx -w